In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# Goose CNN agent (exp007, Track G of SPEC_4WEEKS).
# Inlined Kaggle copy of agents/goose_cnn_model.py + agents/goose_cnn_agent.py
# (the local source-of-truth). Lazy-imports torch and gracefully falls back
# to the trigger_bfs-style state-graph + non-bg-coord-prior behavior if
# torch / CUDA is unavailable in the Kaggle runtime.
# =====================================================================
from __future__ import annotations

import contextlib
import hashlib
import logging
import random as _random
from collections import OrderedDict, deque
from dataclasses import dataclass, field
from typing import Any

import numpy as np

from agents.agent import Agent
from arcengine import GameAction, GameState

logger = logging.getLogger(__name__)

GRID_SIZE = 64
NUM_COLORS = 16
NUM_SIMPLE_ACTIONS = 5
MAX_BUFFER = 50000

# --------------------------------------------------------------------
# State graph (mirror of agents/state_graph.py)
# --------------------------------------------------------------------


def _hash_frame(frame_layers) -> bytes:
    h = hashlib.blake2b(digest_size=8)
    if frame_layers is None:
        return h.digest()
    for layer in frame_layers:
        tobytes = getattr(layer, "tobytes", None)
        if callable(tobytes):
            h.update(tobytes())
        else:
            for row in layer:
                h.update(bytes(int(v) % 256 for v in row))
    return h.digest()


def _hash_grid(grid_2d) -> bytes:
    h = hashlib.blake2b(digest_size=8)
    tobytes = getattr(grid_2d, "tobytes", None)
    if callable(tobytes):
        h.update(tobytes())
    else:
        for row in grid_2d:
            for v in row:
                h.update(int(v).to_bytes(2, "little", signed=False))
    return h.digest()


@dataclass
class StateNode:
    state_hash: bytes
    visit_count: int = 0
    untried_actions: set = field(default_factory=set)
    edges: dict = field(default_factory=dict)
    incoming_change_score: float = 0.0


class StateGraph:
    def __init__(self):
        self.nodes: dict = {}
        self.frontier: deque = deque()
        self.current_levels: int = 0

    def reset(self):
        self.nodes.clear()
        self.frontier.clear()

    def maybe_reset_for_level(self, levels):
        if levels != self.current_levels:
            self.reset()
            self.current_levels = levels
            return True
        return False

    def add_or_get(self, state_hash, available_actions, levels):
        node = self.nodes.get(state_hash)
        if node is None:
            untried = {int(a) for a in (available_actions or []) if int(a) != 0}
            node = StateNode(state_hash=state_hash, untried_actions=untried)
            self.nodes[state_hash] = node
            if untried:
                self.frontier.append(state_hash)
        return node

    def observe(self, prev_hash, action_id, next_hash, change_score=0.0):
        if prev_hash is not None and prev_hash in self.nodes:
            node = self.nodes[prev_hash]
            node.edges[action_id] = next_hash
            node.untried_actions.discard(action_id)
            if not node.untried_actions:
                with contextlib.suppress(ValueError):
                    self.frontier.remove(prev_hash)
        if next_hash in self.nodes:
            self.nodes[next_hash].visit_count += 1
            self.nodes[next_hash].incoming_change_score = change_score


# --------------------------------------------------------------------
# Experience buffer (mirror of ExperienceBuffer)
# --------------------------------------------------------------------


class ExperienceBuffer:
    def __init__(self, max_size=MAX_BUFFER):
        self.max_size = max_size
        self._data = OrderedDict()

    def __len__(self):
        return len(self._data)

    def add(self, state_hash, action_id, frame_changed, click_xy=None):
        x, y = (click_xy or (-1, -1))
        key = (state_hash, int(action_id), int(x), int(y))
        if key in self._data:
            self._data.move_to_end(key)
            self._data[key] = int(bool(frame_changed))
            return
        if len(self._data) >= self.max_size:
            self._data.popitem(last=False)
        self._data[key] = int(bool(frame_changed))

    def clear(self):
        self._data.clear()

    def sample(self, batch_size, rng):
        if not self._data:
            return []
        keys = list(self._data.keys())
        n = min(batch_size, len(keys))
        picked = rng.sample(keys, n)
        return [(k[0], k[1], k[2], k[3], self._data[k]) for k in picked]


# --------------------------------------------------------------------
# Torch model (lazy)
# --------------------------------------------------------------------


def _encode_one_hot(grid_2d):
    import torch

    arr = grid_2d if isinstance(grid_2d, np.ndarray) else np.asarray(grid_2d, dtype=np.int64)
    arr = np.clip(arr, 0, NUM_COLORS - 1)
    onehot = np.zeros((NUM_COLORS, GRID_SIZE, GRID_SIZE), dtype=np.float32)
    for c in range(NUM_COLORS):
        onehot[c] = (arr == c).astype(np.float32)
    return torch.from_numpy(onehot)


def _make_torch_model():
    from torch import nn

    class GooseCNN(nn.Module):
        def __init__(self):
            super().__init__()
            ch = [NUM_COLORS, 32, 64, 128, 256]
            blocks = []
            for i in range(4):
                blocks.append(nn.Conv2d(ch[i], ch[i + 1], kernel_size=3, padding=1, bias=False))
                blocks.append(nn.BatchNorm2d(ch[i + 1]))
                blocks.append(nn.ReLU(inplace=True))
            self.backbone = nn.Sequential(*blocks)
            self.action_head = nn.Linear(256, NUM_SIMPLE_ACTIONS)
            self.coord_head = nn.Conv2d(256, 1, kernel_size=1)

        def forward(self, x):
            feat = self.backbone(x)
            pooled = feat.mean(dim=(2, 3))
            return self.action_head(pooled), self.coord_head(feat).squeeze(1)

    return GooseCNN()


class GooseCNNPredictor:
    def __init__(self, seed=0, max_buffer=MAX_BUFFER, batch_size=32, lr=5e-4, ent=0.001):
        self._seed = seed
        self._batch_size = batch_size
        self._lr = lr
        self._ent = ent
        self._rng = _random.Random(seed)
        self.buffer = ExperienceBuffer(max_size=max_buffer)
        self._torch = None
        self._model = None
        self._optim = None
        self.device = "cpu"
        self.available = False
        try:
            import torch

            self._torch = torch
            torch.manual_seed(seed)
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self._build()
            self.available = True
        except Exception as e:
            logger.warning("GooseCNN: torch unavailable (%s); falling back to uniform.", e)
            self.available = False

    def _build(self):
        torch = self._torch
        self._model = _make_torch_model().to(self.device)
        self._optim = torch.optim.Adam(self._model.parameters(), lr=self._lr)

    def reset(self, seed=None):
        self.buffer.clear()
        if seed is not None:
            self._seed = seed
            self._rng = _random.Random(seed)
        if self._torch is None:
            return
        self._torch.manual_seed(self._seed)
        self._build()

    def predict(self, grid_2d):
        if not self.available:
            return (
                np.full((NUM_SIMPLE_ACTIONS,), 0.5, dtype=np.float32),
                np.full((GRID_SIZE, GRID_SIZE), 0.5, dtype=np.float32),
            )
        torch = self._torch
        self._model.eval()
        with torch.no_grad():
            x = _encode_one_hot(grid_2d).unsqueeze(0).to(self.device)
            al, cl = self._model(x)
            ap = torch.sigmoid(al).squeeze(0).detach().cpu().numpy()
            cp = torch.sigmoid(cl).squeeze(0).detach().cpu().numpy()
        return ap.astype(np.float32), cp.astype(np.float32)

    def update(self, n_steps, grids_by_hash):
        if not self.available or n_steps <= 0 or len(self.buffer) < self._batch_size:
            return
        torch = self._torch
        from torch.nn import functional as nn_func

        self._model.train()
        for _ in range(n_steps):
            samples = self.buffer.sample(self._batch_size, self._rng)
            xs, at, am, ct, cm = [], [], [], [], []
            for sh, aid, x, y, label in samples:
                grid = grids_by_hash.get(sh)
                if grid is None:
                    continue
                xs.append(_encode_one_hot(grid))
                _at = [0.0] * NUM_SIMPLE_ACTIONS
                _am = [0.0] * NUM_SIMPLE_ACTIONS
                _ct = [[0.0] * GRID_SIZE for _ in range(GRID_SIZE)]
                _cm = [[0.0] * GRID_SIZE for _ in range(GRID_SIZE)]
                if 1 <= aid <= NUM_SIMPLE_ACTIONS:
                    _at[aid - 1] = float(label)
                    _am[aid - 1] = 1.0
                elif aid == 6 and 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE:
                    _ct[y][x] = float(label)
                    _cm[y][x] = 1.0
                at.append(_at)
                am.append(_am)
                ct.append(_ct)
                cm.append(_cm)
            if not xs:
                continue
            xb = torch.stack(xs, dim=0).to(self.device)
            t_at = torch.tensor(at, dtype=torch.float32, device=self.device)
            t_am = torch.tensor(am, dtype=torch.float32, device=self.device)
            t_ct = torch.tensor(ct, dtype=torch.float32, device=self.device)
            t_cm = torch.tensor(cm, dtype=torch.float32, device=self.device)
            al, cl = self._model(xb)
            la = nn_func.binary_cross_entropy_with_logits(al, t_at, reduction="none")
            lc = nn_func.binary_cross_entropy_with_logits(cl, t_ct, reduction="none")
            la = (la * t_am).sum() / (t_am.sum() + 1e-6)
            lc = (lc * t_cm).sum() / (t_cm.sum() + 1e-6)
            ap = torch.sigmoid(al)
            ent = -(ap * torch.log(ap + 1e-6) + (1 - ap) * torch.log(1 - ap + 1e-6)).mean()
            loss = la + lc - self._ent * ent
            self._optim.zero_grad()
            loss.backward()
            self._optim.step()


# --------------------------------------------------------------------
# Sampling helpers
# --------------------------------------------------------------------


def _layers_to_grid(layers):
    if not layers:
        return None
    g = layers[-1]
    g = g if isinstance(g, np.ndarray) else np.asarray(g, dtype=np.uint8)
    if getattr(g, "ndim", 0) != 2:
        return None
    return g


def _frame_changed(prev_layers, next_layers):
    p = _layers_to_grid(prev_layers)
    n = _layers_to_grid(next_layers)
    if p is None or n is None:
        return False
    if p.shape != n.shape:
        return True
    return bool((p != n).any())


def _non_bg_mask(layers):
    g = _layers_to_grid(layers)
    if g is None:
        return None
    bg = int(np.bincount(g.flatten(), minlength=16).argmax())
    return (g != bg).astype(np.float64)


def _softmax_sample_2d(probs_2d, rng, temperature=0.7, non_bg_mask=None):
    arr = np.asarray(probs_2d, dtype=np.float64)
    if arr.shape != (GRID_SIZE, GRID_SIZE):
        return (rng.randint(0, GRID_SIZE - 1), rng.randint(0, GRID_SIZE - 1))
    logits = np.log(np.clip(arr, 1e-6, 1.0)) / max(temperature, 1e-3)
    if non_bg_mask is not None:
        m = np.asarray(non_bg_mask, dtype=np.float64)
        if m.shape == (GRID_SIZE, GRID_SIZE) and m.sum() > 0:
            logits = np.where(m > 0, logits, -1e9)
    logits = logits - logits.max()
    p = np.exp(logits)
    p = p / max(p.sum(), 1e-12)
    flat = p.flatten()
    cum = np.cumsum(flat)
    idx = int(np.searchsorted(cum, rng.random()))
    idx = min(idx, GRID_SIZE * GRID_SIZE - 1)
    y, x = divmod(idx, GRID_SIZE)
    return (int(x), int(y))


def _softmax_sample_1d(logits, avail_mask, rng, temperature=1.0):
    a = np.asarray(logits, dtype=np.float64)
    m = np.asarray(avail_mask, dtype=np.float64)
    a = a / max(temperature, 1e-3)
    a = a - a.max()
    p = np.exp(a) * m
    s = p.sum()
    if s <= 0:
        idxs = np.where(m > 0)[0]
        if len(idxs) == 0:
            return 0
        return int(rng.choice(idxs.tolist()))
    p = p / s
    cum = np.cumsum(p)
    idx = int(np.searchsorted(cum, rng.random()))
    return min(idx, len(p) - 1)


def _action_logits_to_six(ap, cp):
    six = np.zeros((6,), dtype=np.float32)
    six[:NUM_SIMPLE_ACTIONS] = np.asarray(ap, dtype=np.float32)[:NUM_SIMPLE_ACTIONS]
    six[NUM_SIMPLE_ACTIONS] = float(np.max(cp)) if cp is not None else 0.5
    return six


# --------------------------------------------------------------------
# MyAgent (Kaggle entry point)
# --------------------------------------------------------------------


class MyAgent(Agent):
    """StochasticGoose CNN-driven agent with state-graph dedup + non-bg coord prior."""

    MAX_ACTIONS = 80

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._rng = _random.Random(0)
        self._train_every = 4
        self._train_steps = 4
        self._action_temp = 1.0
        self._coord_temp = 0.7
        self.predictor = GooseCNNPredictor(seed=0)
        self.graph = StateGraph()
        self._step = 0
        self._prev_hash = None
        self._prev_action = None
        self._prev_xy = None
        self._prev_layers = None
        self._prev_levels = 0
        self._grids_by_hash: dict = {}

    def is_done(self, frames, latest_frame) -> bool:
        return latest_frame.state == GameState.WIN

    def _on_level_transition(self):
        self.predictor.reset(seed=self._rng.randint(0, 1_000_000))
        self.graph = StateGraph()
        self._grids_by_hash.clear()
        self._prev_hash = None
        self._prev_action = None
        self._prev_xy = None
        self._prev_layers = None
        self._step = 0

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self._prev_hash = None
            self._prev_action = None
            self._prev_xy = None
            self._prev_layers = None
            return GameAction.RESET

        cur_layers = list(getattr(latest_frame, "frame", []) or [])
        cur_hash = _hash_frame(cur_layers) if cur_layers else b"\x00" * 8
        cur_levels = int(getattr(latest_frame, "levels_completed", 0))

        if cur_levels != self._prev_levels:
            self._on_level_transition()
            self._prev_levels = cur_levels

        avail = list(getattr(latest_frame, "available_actions", []) or [1, 2, 3, 4, 5, 6, 7])
        self.graph.add_or_get(cur_hash, available_actions=avail, levels=cur_levels)

        cur_grid = _layers_to_grid(cur_layers)
        if cur_grid is not None:
            self._grids_by_hash[_hash_grid(cur_grid)] = cur_grid

        if self._prev_hash is not None and self._prev_action is not None:
            changed = _frame_changed(self._prev_layers, cur_layers)
            self.graph.observe(
                self._prev_hash,
                self._prev_action,
                cur_hash,
                change_score=1.0 if changed else 0.0,
            )
            if self._prev_layers is not None:
                pg = _layers_to_grid(self._prev_layers)
                if pg is not None:
                    pgk = _hash_grid(pg)
                    self._grids_by_hash[pgk] = pg
                    self.predictor.buffer.add(
                        pgk, self._prev_action, changed, click_xy=self._prev_xy
                    )

        if self._step > 0 and self._step % self._train_every == 0 and self.predictor.available:
            self.predictor.update(self._train_steps, self._grids_by_hash)

        if cur_grid is not None:
            ap, cp = self.predictor.predict(cur_grid)
        else:
            ap = np.full((NUM_SIMPLE_ACTIONS,), 0.5, dtype=np.float32)
            cp = np.full((GRID_SIZE, GRID_SIZE), 0.5, dtype=np.float32)

        non_reset = [int(a) for a in avail if int(a) != 0]
        if not non_reset:
            non_reset = [1, 2, 3, 4, 5, 6, 7]
        avail_mask = [1.0 if (i + 1) in non_reset else 0.0 for i in range(NUM_SIMPLE_ACTIONS)] + [
            1.0 if 6 in non_reset else 0.0
        ]

        node = self.graph.nodes.get(cur_hash)
        untried_set = set(node.untried_actions) if node is not None else set()
        untried_avail = [a for a in untried_set if a in non_reset]
        nb_mask = _non_bg_mask(cur_layers)

        xy = None
        if untried_avail:
            chosen = self._rng.choice(untried_avail)
            if chosen == 6:
                xy = _softmax_sample_2d(
                    cp, self._rng, temperature=self._coord_temp, non_bg_mask=nb_mask
                )
        else:
            six = _action_logits_to_six(ap, cp)
            if node is not None:
                for a, sh in node.edges.items():
                    if a not in non_reset:
                        continue
                    succ = self.graph.nodes.get(sh)
                    if succ is None:
                        continue
                    boost = float(succ.incoming_change_score)
                    if 1 <= a <= NUM_SIMPLE_ACTIONS:
                        six[a - 1] += boost
                    elif a == 6:
                        six[NUM_SIMPLE_ACTIONS] += boost
            idx = _softmax_sample_1d(six, avail_mask, self._rng, temperature=self._action_temp)
            if idx < NUM_SIMPLE_ACTIONS:
                chosen = idx + 1
            elif 6 in non_reset:
                chosen = 6
                xy = _softmax_sample_2d(
                    cp, self._rng, temperature=self._coord_temp, non_bg_mask=nb_mask
                )
            else:
                chosen = self._rng.choice(non_reset)
                if chosen == 6:
                    xy = _softmax_sample_2d(
                        cp, self._rng, temperature=self._coord_temp, non_bg_mask=nb_mask
                    )

        action = GameAction.from_id(chosen)
        data: dict = {}
        if action.is_complex():
            if xy is None:
                if nb_mask is not None and nb_mask.sum() > 0:
                    ys, xs = np.where(nb_mask > 0)
                    i = self._rng.randrange(len(xs))
                    xy = (int(xs[i]), int(ys[i]))
                else:
                    xy = (
                        self._rng.randint(0, GRID_SIZE - 1),
                        self._rng.randint(0, GRID_SIZE - 1),
                    )
            data = {"x": int(xy[0]), "y": int(xy[1])}
            action.set_data(data)

        self._prev_hash = cur_hash
        self._prev_action = chosen
        self._prev_xy = xy
        self._prev_layers = cur_layers
        self._prev_levels = cur_levels
        self._step += 1
        return action


this only runs if you submit to the competition, not when you do tests

In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py','w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
""")
    with open('/kaggle/working/ARC-AGI-3-Agents/.env','w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet',index=False)

This is a dummy submission fallback, important to keep